In [ ]:
#!pip install -q pypdf google-genai scikit-learn

In [ ]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 10.6 MB/s eta 0:00:00


In [ ]:
from pypdf import PdfReader
from sklearn.metrics.pairwise import cosine_similarity
from google import genai
from google.colab import files
from google.colab import userdata
import numpy as np



In [ ]:
GEMINI_API_KEY=userdata.get('gemini9')

In [ ]:
client=genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
uploaded = files.upload()

pdf = list(uploaded.keys())[0]

text = ""

for page in PdfReader(pdf).pages:
    text += page.extract_text() + "\n"

print(text[:2000])

Saving 01_Employee_Leave_Policy.pdf to 01_Employee_Leave_Policy.pdf
Saving 02_Work_From_Home_Policy.pdf to 02_Work_From_Home_Policy.pdf
Saving 03_Attendance_and_Working_Hours_Policy.pdf to 03_Attendance_and_Working_Hours_Policy.pdf
Saving 04_Travel_and_Reimbursement_Policy.pdf to 04_Travel_and_Reimbursement_Policy.pdf
Sample HR Policy Document | For RAG and training demonstration
Page 1
 EMPLOYEE LEAVE POLICY
Sample Organization | Human Resources
1. Purpose
This policy defines leave entitlements, eligibility, approval requirements, and procedures for employees.
2. Scope
This policy applies to all regular full-time and part-time employees unless an employment agreement or applicable
law provides otherwise.
3. Annual Leave
Eligible employees receive 24 working days of annual leave per calendar year. Leave should normally be
requested at least three working days in advance.
4. Sick Leave
Employees may take up to 12 working days of sick leave per calendar year. A medical certificate may be

In [ ]:
words = text.split()

chunks = [
    " ".join(words[i:i+300])
    for i in range(0, len(words), 300)
]

print(chunks)

['Sample HR Policy Document | For RAG and training demonstration Page 1 EMPLOYEE LEAVE POLICY Sample Organization | Human Resources 1. Purpose This policy defines leave entitlements, eligibility, approval requirements, and procedures for employees. 2. Scope This policy applies to all regular full-time and part-time employees unless an employment agreement or applicable law provides otherwise. 3. Annual Leave Eligible employees receive 24 working days of annual leave per calendar year. Leave should normally be requested at least three working days in advance. 4. Sick Leave Employees may take up to 12 working days of sick leave per calendar year. A medical certificate may be requested for extended or repeated absence. 5. Casual Leave Employees may take up to 6 working days of casual leave per calendar year for personal or urgent matters. 6. Maternity and Paternity Leave Eligible employees receive leave according to company rules and applicable statutory requirements. HR will confirm the 

In [ ]:
embeddings = []

for chunk in chunks:
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=chunk
    )
    embeddings.append(result.embeddings[0].values)

embeddings = np.array(embeddings)



In [ ]:
embeddings

array([[-0.01297573,  0.03996156,  0.01115975, ...,  0.01429105,
         0.01123664,  0.00455862]])

In [ ]:
question = input("Ask HR question: ")

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=question
)

q_embedding = np.array(result.embeddings[0].values)

scores = cosine_similarity(
    [q_embedding],
    embeddings
)[0]

top = scores.argsort()[-3:][::-1]

context = "\n\n".join(chunks[i] for i in top)

print(context)

Ask HR question: what is leave policy
Sample HR Policy Document | For RAG and training demonstration Page 1 EMPLOYEE LEAVE POLICY Sample Organization | Human Resources 1. Purpose This policy defines leave entitlements, eligibility, approval requirements, and procedures for employees. 2. Scope This policy applies to all regular full-time and part-time employees unless an employment agreement or applicable law provides otherwise. 3. Annual Leave Eligible employees receive 24 working days of annual leave per calendar year. Leave should normally be requested at least three working days in advance. 4. Sick Leave Employees may take up to 12 working days of sick leave per calendar year. A medical certificate may be requested for extended or repeated absence. 5. Casual Leave Employees may take up to 6 working days of casual leave per calendar year for personal or urgent matters. 6. Maternity and Paternity Leave Eligible employees receive leave according to company rules and applicable statutor

In [ ]:
prompt = f"""
Answer the question using only the context.

Context:
{context}

Question:
{question}

If the answer is not in the context,
say "Not found in the HR documents."

Answer:
"""

answer = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print("\nANSWER:\n")
print(answer.text)


ANSWER:

Based on the provided context, the leave policy defines leave entitlements, eligibility, approval requirements, and procedures for employees.
